# 03 Color Event Detector Summary + Config Draft v1

18번 실험의 목적은 traffic light와 final redline을 OpenCV 색 + contour 기반으로 검출할 수 있는지 확인하는 것이었다.

이 노트북은 새 실험을 많이 추가하지 않는다. 지금까지의 결과를 정리하고, Pi runtime에 옮길 수 있는 **config 초안**을 만든다.

```text
01d: traffic green/red event 후보 정리
02 : final redline의 색 + horizontal contour 가능성 확인
02a: bottom ROI 기반 final redline event 후보 정리
```

현재는 map에서 event timing까지 검증하지 못했으므로, 결과물은 final contract가 아니라 `draft_v1`이다.


## 18번 실험의 결론

현재 결론은 다음처럼 본다.

```text
traffic green:
  green_conservative HSV + event_near contour + green_slot ROI
  offline 이미지 기준 target hit 약 90%, cross/negative false 0%

traffic red:
  red_balanced HSV + event_near contour + red_slot_strict_x ROI
  offline 이미지 기준 target hit 약 64%, cross/negative false 0%
  단, 실제 runtime에서는 YOLO stop sign event가 red traffic보다 우선해야 한다.

final redline:
  red_balanced HSV + bottom70 ROI + event_soft horizontal contour
  traffic false 0%, 가까운 redline trigger 후보로 적절
  stop_sign_negative hit는 높지만, overlay상 대부분 stop sign 자체가 아니라 바닥 redline-like 영역을 잡은 것이다.
```

중요한 한계:

```text
18번은 offline image 탐색이다.
temporal voting, stop delay, 실제 정지 위치는 map 실험에서 조정해야 한다.
```


In [1]:
from pathlib import Path
from IPython.display import display
import json
import pandas as pd

PROJECT_ROOT = Path(r'~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization')
EXP_ROOT = PROJECT_ROOT / '10_experiments' / '18_opencv_color_event_detector'
OUT_ROOT = EXP_ROOT / 'review_outputs' / '03_color_event_detector_summary_config_draft_v1'
TABLE_DIR = OUT_ROOT / 'tables'
CONFIG_DIR = OUT_ROOT / 'config'
for p in [TABLE_DIR, CONFIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TRAFFIC_TABLE_DIR = EXP_ROOT / 'review_outputs' / '01d_traffic_event_trigger_candidate_v2' / 'tables'
REDLINE_TABLE_DIR = EXP_ROOT / 'review_outputs' / '02a_redline_bottom_roi_event_probe_v1' / 'tables'

print('EXP_ROOT:', EXP_ROOT)
print('OUT_ROOT:', OUT_ROOT)
print('TRAFFIC_TABLE_DIR:', TRAFFIC_TABLE_DIR)
print('REDLINE_TABLE_DIR:', REDLINE_TABLE_DIR)


EXP_ROOT: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\18_opencv_color_event_detector
OUT_ROOT: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\18_opencv_color_event_detector\review_outputs\03_color_event_detector_summary_config_draft_v1
TRAFFIC_TABLE_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\18_opencv_color_event_detector\review_outputs\01d_traffic_event_trigger_candidate_v2\tables
REDLINE_TABLE_DIR: ~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\18_opencv_color_event_detector\review_outputs\02a_redline_bottom_roi_event_probe_v1\tables


## 근거 표 로드

03은 앞 노트북들의 summary CSV를 직접 읽어서 선택 후보를 보여준다. 그래야 config 초안이 어디서 나온 값인지 추적 가능하다.


In [2]:
traffic_judgement = pd.read_csv(TRAFFIC_TABLE_DIR / 'traffic_event_v2_candidate_judgement.csv')
redline_ranking = pd.read_csv(REDLINE_TABLE_DIR / 'bottom_roi_event_candidate_ranking.csv')

selected_traffic = traffic_judgement[traffic_judgement['detector_name'].isin([
    'green__green_conservative__event_near__green_slot',
    'red__red_balanced__event_near__red_slot_strict_x',
])].copy()

selected_redline = redline_ranking[
    (redline_ranking['profile'] == 'red_balanced_s100_v80') &
    (redline_ranking['roi'] == 'bottom70') &
    (redline_ranking['shape'].isin(['event_soft', 'event_balanced']))
].copy()

selected_traffic.to_csv(TABLE_DIR / 'selected_traffic_candidates.csv', index=False, encoding='utf-8-sig')
selected_redline.to_csv(TABLE_DIR / 'selected_redline_candidates.csv', index=False, encoding='utf-8-sig')

display(selected_traffic[[
    'detector_name','target','color_profile','contour_profile','roi_profile',
    'target_hit_pct','cross_color_hit_pct','stop_sign_negative_hit_pct','final_redline_negative_hit_pct','judgement'
]])

display(selected_redline[[
    'profile','roi','roi_y0','shape','final_redline','stop_sign_negative','traffic_red','traffic_green','event_score_probe'
]])


,detector_name,target,color_profile,contour_profile,roi_profile,target_hit_pct,cross_color_hit_pct,stop_sign_negative_hit_pct,final_redline_negative_hit_pct,judgement
0,green__green_conservative__event_near__green_slot,green,green_conservative,event_near,green_slot,90.415335,0.0,0.0,0.0,green_contract_candidate
8,red__red_balanced__event_near__red_slot_strict_x,red,red_balanced,event_near,red_slot_strict_x,63.556851,0.0,0.0,0.0,red_candidate_needs_yolo_stop_veto


,profile,roi,roi_y0,shape,final_redline,stop_sign_negative,traffic_red,traffic_green,event_score_probe
0,red_balanced_s100_v80,bottom70,0.7,event_soft,59.405941,43.776824,0.0,0.0,33.139846
1,red_balanced_s100_v80,bottom70,0.7,event_balanced,53.465347,43.776824,0.0,0.0,27.199252


## Draft config의 의미

아래 config는 Pi runtime에 그대로 넣을 수 있는 형태를 의식해서 작성한다.

하지만 두 종류의 값은 아직 최종값이 아니다.

```text
temporal_voting:
  실제 카메라 stream에서 몇 프레임 연속 감지해야 이벤트로 인정할지는 현장 튜닝 필요.

redline stop_delay_sec:
  차량 속도, 모터 설정, 감지 시점에 따라 조정 필요.
```

따라서 이 파일의 상태는 `draft_v1_offline_image_based`다.


In [3]:
color_event_config_draft = {
    'version': 'draft_v1_offline_image_based',
    'source_experiment': '18_opencv_color_event_detector',
    'created_from_notebooks': [
        '01d_traffic_event_trigger_candidate_v2.ipynb',
        '02_redline_color_contour_probe_v1.ipynb',
        '02a_redline_bottom_roi_event_probe_v1.ipynb',
        '03_color_event_detector_summary_config_draft_v1.ipynb',
    ],
    'input_assumptions': {
        'camera_frame': 'BGR image, corrected orientation/color as used in drive_test after camera fix',
        'image_size_observed': [1296, 972],
        'roi_coordinates': 'normalized image coordinates unless stated otherwise',
    },
    'runtime_notes': {
        'purpose': 'event hints for driving state machine, not pixel-perfect object detection',
        'traffic_stop_priority': 'YOLO stop sign event should have priority/veto over traffic red HSV event.',
        'redline_timing': 'final stop timing must be tuned on map using stop_delay_sec and motor speed.',
        'temporal_voting': 'draft values; verify with live camera stream.',
    },
    'traffic_light': {
        'green': {
            'enabled': True,
            'decision': 'provisional',
            'hsv_profile': {
                'name': 'green_conservative',
                'ranges': [
                    {'lower': [65, 130, 115], 'upper': [100, 255, 255]},
                ],
            },
            'contour_profile': {
                'name': 'event_near',
                'min_area': 1400,
                'min_width_px': 32,
                'min_height_px': 35,
                'max_aspect_wh': 4.0,
                'min_fill': 0.045,
            },
            'roi_profile': {
                'name': 'green_slot',
                'min_center_x': 0.58,
                'max_center_y': 0.38,
            },
            'temporal_voting': {
                'window_frames': 5,
                'required_hits': 3,
                'note': 'draft; reduce to 2/3 if event is too late on Pi.',
            },
            'offline_reference': {
                'target_hit_pct': 90.415335,
                'cross_color_hit_pct': 0.0,
                'stop_sign_negative_hit_pct': 0.0,
                'final_redline_negative_hit_pct': 0.0,
            },
        },
        'red': {
            'enabled': True,
            'decision': 'provisional_with_yolo_stop_veto',
            'hsv_profile': {
                'name': 'red_balanced',
                'ranges': [
                    {'lower': [0, 105, 90], 'upper': [12, 255, 255]},
                    {'lower': [168, 105, 90], 'upper': [179, 255, 255]},
                ],
            },
            'contour_profile': {
                'name': 'event_near',
                'min_area': 180,
                'min_width_px': 13,
                'min_height_px': 15,
                'max_aspect_wh': 5.0,
                'min_fill': 0.035,
            },
            'roi_profile': {
                'name': 'red_slot_strict_x',
                'min_center_x': 0.70,
                'min_center_y': 0.32,
                'max_center_y': 0.52,
            },
            'temporal_voting': {
                'window_frames': 5,
                'required_hits': 3,
                'note': 'draft; use YOLO stop sign priority if both stop sign and red traffic are active.',
            },
            'offline_reference': {
                'target_hit_pct': 63.556851,
                'cross_color_hit_pct': 0.0,
                'stop_sign_negative_hit_pct': 0.0,
                'final_redline_negative_hit_pct': 0.0,
            },
        },
    },
    'final_redline': {
        'enabled': True,
        'decision': 'provisional_bottom_roi_event_trigger',
        'hsv_profile': {
            'name': 'red_balanced_s100_v80',
            'ranges': [
                {'lower': [0, 100, 80], 'upper': [12, 255, 255]},
                {'lower': [168, 100, 80], 'upper': [179, 255, 255]},
            ],
        },
        'mask_postprocess': {
            'open_kernel': [3, 3],
            'close_kernel': [3, 3],
            'horizontal_close_kernel': [17, 3],
        },
        'bottom_roi': {
            'name': 'bottom70',
            'y0': 0.70,
            'note': 'Use y >= 0.70H as final redline event zone. Tune to bottom75 if trigger is too early.',
        },
        'shape_profile': {
            'name': 'event_soft',
            'min_area': 350,
            'min_width_ratio': 0.07,
            'min_aspect_wh': 2.0,
            'min_center_overlap': 0.10,
            'min_height_px': 3,
            'central_band': [0.35, 0.65],
        },
        'temporal_voting': {
            'window_frames': 3,
            'required_hits': 2,
            'note': 'draft; increase to 3/5 if false triggers appear.',
        },
        'event_action': {
            'on_detected': 'start_stop_timer',
            'stop_delay_sec_default': 0.8,
            'stop_delay_sec_note': 'Tune on map based on motor speed. This is not validated offline.',
        },
        'offline_reference': {
            'final_redline_hit_pct': 59.405941,
            'traffic_red_hit_pct': 0.0,
            'traffic_green_hit_pct': 0.0,
            'stop_sign_negative_hit_pct': 43.776824,
            'stop_sign_negative_note': 'Overlay shows many hits are bottom floor redline-like regions in stop-sign images, not the stop sign itself.',
        },
    },
    'event_arbitration_draft': [
        'final_redline_event can force final stop when enabled by mission state.',
        'YOLO stop sign event has priority over HSV traffic red/green if both are active.',
        'Traffic green/red should only be consumed in traffic-light state, not globally.',
        'HSV detectors should output observations; the driving state machine decides actions.',
    ],
}

config_path = CONFIG_DIR / 'color_event_detector_config_draft_v1.json'
config_path.write_text(json.dumps(color_event_config_draft, ensure_ascii=False, indent=2), encoding='utf-8')
print(config_path)
print(config_path.read_text(encoding='utf-8')[:2500])


~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\18_opencv_color_event_detector\review_outputs\03_color_event_detector_summary_config_draft_v1\config\color_event_detector_config_draft_v1.json
{
  "version": "draft_v1_offline_image_based",
  "source_experiment": "18_opencv_color_event_detector",
  "created_from_notebooks": [
    "01d_traffic_event_trigger_candidate_v2.ipynb",
    "02_redline_color_contour_probe_v1.ipynb",
    "02a_redline_bottom_roi_event_probe_v1.ipynb",
    "03_color_event_detector_summary_config_draft_v1.ipynb"
  ],
  "input_assumptions": {
    "camera_frame": "BGR image, corrected orientation/color as used in drive_test after camera fix",
    "image_size_observed": [
      1296,
      972
    ],
    "roi_coordinates": "normalized image coordinates unless stated otherwise"
  },
  "runtime_notes": {
    "purpose": "event hints for driving state machine, not pixel-perfect object detection",
    "traffic_stop_priority": "YOLO stop sign e

## Pi runtime에 옮길 때의 구현 형태

Pi 쪽에서는 아래처럼 하나의 OpenCV worker에서 traffic과 redline을 같이 처리하면 된다.

```text
input: BGR camera frame
  ├─ traffic_green detector
  ├─ traffic_red detector
  └─ final_redline detector

output observation:
  {
    "traffic_green_seen": bool,
    "traffic_red_seen": bool,
    "final_redline_seen": bool,
    "debug": { bbox, area, roi, consecutive_count }
  }
```

이 worker는 바로 motor를 제어하지 않는다. motor action은 상위 state machine이 결정해야 한다.

```text
lane model: steering
YOLO sign: direction / stop / speed signs
OpenCV event: traffic light / final redline
state machine: 어떤 event를 지금 사용할지 결정
```


## 18번의 남은 한계

이 실험은 충분히 의미 있지만, 아직 아래는 검증하지 않았다.

```text
1. live camera stream에서 temporal voting이 안정적인가?
2. final redline 감지 후 stop_delay_sec가 실제 정차 위치와 맞는가?
3. traffic red/green event가 실제 state machine 타이밍에서 너무 늦거나 빠르지 않은가?
4. lane + YOLO + OpenCV event를 동시에 돌렸을 때 latency budget이 맞는가?
```

따라서 18번의 결론은 다음 정도로 둔다.

```text
OpenCV 색 + contour 기반 traffic/redline detector는 구현 가능하다.
offline image 기준 provisional config 초안까지 도출했다.
최종 event timing은 map 실험에서 state machine과 함께 검증한다.
```
